 UCI 477 Real Estate Valuation Pipeline

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVR
from ucimlrepo import fetch_ucirepo

RANDOM_STATE = 42
TEST_SIZE = 1 / 3
CV_FOLDS = 5
CV_SPLITTER = KFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
CV_SELECT_METRIC = "cv_rmse"
CV_SCORING = {"mae": "neg_mean_absolute_error", "rmse": "neg_root_mean_squared_error", "r2": "r2"}
X3_COL = "X3 distance to the nearest MRT station"
pd.set_option("display.width", 10000)
pd.set_option("display.max_columns", None)


In [2]:
ds = fetch_ucirepo(id=477)
print("features:", ds.data.features.columns.tolist())
print("targets:", ds.data.targets.columns.tolist())
print("ids:", ds.data.ids.columns.tolist())
X = ds.data.features.copy()
y = ds.data.targets.iloc[:, 0]


features: ['X1 transaction date', 'X2 house age', 'X3 distance to the nearest MRT station', 'X4 number of convenience stores', 'X5 latitude', 'X6 longitude']
targets: ['Y house price of unit area']
ids: ['No']


In [3]:
df = X.copy()
df["Y"] = y.values
print("shape:", df.shape)
print("missing:", df.isna().sum().sum())
print("duplicates:", df.duplicated().sum())
print("describe:\n", df.describe())


shape: (414, 7)
missing: 0
duplicates: 0
describe:
        X1 transaction date  X2 house age  X3 distance to the nearest MRT station  X4 number of convenience stores  X5 latitude  X6 longitude           Y
count           414.000000    414.000000                              414.000000                       414.000000   414.000000    414.000000  414.000000
mean           2013.148971     17.712560                             1083.885689                         4.094203    24.969030    121.533361   37.980193
std               0.281967     11.392485                             1262.109595                         2.945562     0.012410      0.015347   13.606488
min            2012.667000      0.000000                               23.382840                         0.000000    24.932070    121.473530    7.600000
25%            2012.917000      9.025000                              289.324800                         1.000000    24.963000    121.528085   27.700000
50%            2013.167000    

In [4]:
feature_cols = [c for c in df.columns if c != "Y"]
X_all, y_all = df[feature_cols], df["Y"]
X_train, X_test, y_train, y_test = train_test_split(X_all, y_all, test_size=TEST_SIZE, random_state=RANDOM_STATE)
X_train = X_train.copy()
X_test = X_test.copy()
X_train["X3_log"] = np.log1p(X_train[X3_COL])
X_test["X3_log"] = np.log1p(X_test[X3_COL])
feature_sets = {"base": feature_cols, "base_x3log": feature_cols + ["X3_log"]}
print(f"train={X_train.shape[0]} test={X_test.shape[0]}")


train=276 test=138


In [5]:
train_df = X_train.copy()
train_df["Y"] = y_train
print("\ndescribe:\n", train_df.describe())
print("\ncorrelation:\n", train_df.corr())
fig, axes = plt.subplots(2, 4, figsize=(14, 6))
for ax, col in zip(axes.flat, train_df.columns):
    ax.hist(train_df[col], bins=20, edgecolor="black")
    ax.set_title(col, fontsize=8)
for ax in axes.flat[len(train_df.columns):]:
    ax.set_visible(False)
plt.tight_layout()
plt.savefig("eda_distributions.png", dpi=100)
plt.close()
corr = train_df.corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)), corr.columns, rotation=45, ha="right", fontsize=7)
ax.set_yticks(range(len(corr.columns)), corr.columns, fontsize=7)
fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.savefig("eda_correlation.png", dpi=100)
plt.close()
fig, axes = plt.subplots(2, 3, figsize=(10, 6))
for ax, col in zip(axes.flat, feature_cols):
    ax.scatter(train_df[col], train_df["Y"], s=8, alpha=0.6)
    ax.set_title(col, fontsize=8)
plt.tight_layout()
plt.savefig("eda_scatter.png", dpi=100)
plt.close()



describe:
        X1 transaction date  X2 house age  X3 distance to the nearest MRT station  X4 number of convenience stores  X5 latitude  X6 longitude      X3_log           Y
count           276.000000    276.000000                              276.000000                       276.000000   276.000000    276.000000  276.000000  276.000000
mean           2013.169399     17.402174                             1051.099834                         4.217391    24.969698    121.533663    6.371362   38.642754
std               0.280516     11.379599                             1241.491872                         3.010863     0.012495      0.014995    1.085723   14.025988
min            2012.667000      0.000000                               49.661050                         0.000000    24.938850    121.475160    3.925157   11.600000
25%            2012.917000      8.475000                              286.786750                         1.000000    24.963045    121.529805    5.662102   27.30000

In [6]:
models = {
    "baseline_mean": DummyRegressor(strategy="mean"),
    "linear_regression": Pipeline([("scaler", StandardScaler()), ("model", LinearRegression())]),
    "random_forest": RandomForestRegressor(random_state=RANDOM_STATE),
    "gradient_boosting": GradientBoostingRegressor(random_state=RANDOM_STATE),
    "svr": Pipeline([("scaler", StandardScaler()), ("model", SVR())]),
}
cv_rows = []
for feat_name, cols in feature_sets.items():
    Xt = X_train[cols]
    for target_name, use_log in (("Y", False), ("Y_log", True)):
        for model_name, model in models.items():
            est = TransformedTargetRegressor(regressor=model, func=np.log1p, inverse_func=np.expm1) if use_log else model
            s = cross_validate(est, Xt, y_train, cv=CV_SPLITTER, scoring=CV_SCORING, n_jobs=-1)
            cv_rows.append({"features": feat_name, "target": target_name, "model": model_name, "cv_mae": -s["test_mae"].mean(), "cv_rmse": -s["test_rmse"].mean(), "cv_r2": s["test_r2"].mean()})
cv_results = pd.DataFrame(cv_rows).sort_values(CV_SELECT_METRIC, ascending=True)
print(cv_results.round(3).to_string(index=False))


  features target             model  cv_mae  cv_rmse  cv_r2
base_x3log  Y_log               svr   5.221    7.798  0.681
base_x3log  Y_log     random_forest   5.141    7.830  0.676
      base  Y_log               svr   5.225    7.844  0.678
      base  Y_log     random_forest   5.170    7.901  0.670
      base      Y     random_forest   5.223    8.087  0.652
base_x3log  Y_log gradient_boosting   5.334    8.100  0.656
base_x3log      Y linear_regression   5.388    8.124  0.649
      base  Y_log gradient_boosting   5.329    8.129  0.654
base_x3log      Y     random_forest   5.245    8.130  0.649
base_x3log      Y gradient_boosting   5.382    8.196  0.644
      base      Y gradient_boosting   5.402    8.209  0.642
base_x3log  Y_log linear_regression   5.541    8.230  0.643
      base  Y_log linear_regression   5.950    8.737  0.602
      base      Y linear_regression   6.512    9.065  0.569
base_x3log      Y               svr   6.321    9.157  0.563
      base      Y               svr   6.

In [7]:
best = cv_results.iloc[0]
best_cols = feature_sets[best["features"]]
best_model = models[best["model"]]
best_est = TransformedTargetRegressor(regressor=best_model, func=np.log1p, inverse_func=np.expm1) if best["target"] == "Y_log" else best_model
best_est.fit(X_train[best_cols], y_train)
test_pred = best_est.predict(X_test[best_cols])
test_mae = mean_absolute_error(y_test, test_pred)
test_rmse = mean_squared_error(y_test, test_pred) ** 0.5
test_r2 = r2_score(y_test, test_pred)
print(f"best features={best['features']} target={best['target']} model={best['model']} test_mae={test_mae:.3f} test_rmse={test_rmse:.3f} test_r2={test_r2:.3f}")


best features=base_x3log target=Y_log model=svr test_mae=4.599 test_rmse=6.719 test_r2=0.717
